In [ ]:
!pip install pyspark --quiet
print('Pyspark installation complete!')

Pyspark installation complete!


In [ ]:
# import pyspark modules and create sparksession

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# create SparkSession

spark=SparkSession.builder\
.appName('Day4_bigData_Sales')\
.config('spark.sql.adaptive.enable','true')\
.getOrCreate()

print(f'Spark version :{spark.version}')
print(f'SparkSession  :ACTIVE')
print(f'Application   :{spark.sparkContext.appName}')

Spark version :4.0.2
SparkSession  :ACTIVE
Application   :Day4_bigData_Sales


In [ ]:
# load csv into pyspark dataframe

df_bronze=spark.read\
.option('header','true')\
.option('inferSchema','true')\
.csv('large_sales_data.csv')

print('=== BRONZE LAYER - Raw Data===')
print(f'Rows   :{df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
print(f'Name   :{df_bronze.columns}')
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data===
Rows   :5000
Columns:13
Name   :['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
# inspect 1st row and summary statistics

print('First 5 rows:')
df_bronze.show(5,truncate=False)

print('\nBasic statistics for numerical columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [ ]:
# inspect 1st row and summary statistics

print('Last 5 rows:')
df_bronze.toPandas().tail(5)

print('\nBasic statistics for numerical columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

Last 5 rows:

Basic statistics for numerical columns:
+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+



In [ ]:
#save bronze layer as parquet

df_bronze.write\
.mode('overwrite')\
.parquet('sales_bonze.parquet')

print('Bronze Parquet saved:sales_bronze.parquet')

#compare file sizze

import os
def get_dir_size(path):
  """Get total size of a directory in KB."""
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total=0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total+=os.path.getsize(os.path.join(dirpath,f))
  return total/1024

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction =(1-parquet_size/csv_size)*100

print(f'\nCSV size  :{csv_size:.1f}KB')
print(f'Parquet size:{parquet_size:.1f}KB')
print(f'Reduction   :{reduction:.1f}%smaller')
print(f'\nAt 1 TB scale:CSV=1000 GB->Parquet={1000*(1-reduction/100):.0f}GB')

Bronze Parquet saved:sales_bronze.parquet

CSV size  :529.3KB
Parquet size:0.0KB
Reduction   :100.0%smaller

At 1 TB scale:CSV=1000 GB->Parquet=0GB


In [ ]:
df_bronze.filter(F.col("revenue")>50000).show()

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|
|    1004|   Suresh Rao| Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West|  Ravi Kumar|Cash on Delivery|  Processing|
|    1008|  Priya Patel| Laptop|Electronics|      13|     45000| 585000|2023-05-29|  Chennai| South| Meera Patel|     Credit Card|   Can

In [ ]:
df_silver=df_bronze\
.dropDuplicates()\
.dropna(subset=['order_id','product','revenue'])

df_silver=df_silver.withColumn(
    'order_data',
    to_date(col('order_date'),'dd-MM-yyyy')
)

df_silver=df_silver\
.withColumn('order_year',year(col('order_data')))\
    .withColumn('order_month',month(col('order_data')))

df_silver=df_silver.withColumn('revenue_category',
                               F.when(col('revenue')>40000,'High') # Corrected: 'revenu' changed to 'revenue'
                               .when(col('revenue')>10000,'Medium') # Corrected: 'revenu' changed to 'revenue'
                               .otherwise('Low') # Added a default category for values <= 10000
                               )

print(f'Silver layer rows:{df_silver.count()}')
print('New columns adder:order_year.order_month.revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(8)

Silver layer rows:5000
New columns adder:order_year.order_month.revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          Medium|
|    Webcam|  17500|      2023|          1|          Medium|
|   Speaker|  58500|      2023|          4|            High|
|  Keyboard|   9600|      2023|         12|             Low|
|    Laptop| 180000|      2023|          8|            High|
|Headphones|  38500|      2023|          5|          Medium|
|    Webcam|  35000|      2023|         11|          Medium|
|    Laptop| 360000|      2023|          1|            High|
+----------+-------+----------+-----------+----------------+
only showing top 8 rows


In [ ]:
print('Duplicate products and their counts:')
df_bronze.groupBy('product').count().filter(F.col('count') > 1).orderBy(F.col('count').desc()).show(truncate=False)

Duplicate products and their counts:
+----------+-----+
|product   |count|
+----------+-----+
|Webcam    |532  |
|Tablet    |532  |
|USB Hub   |527  |
|Laptop    |502  |
|Keyboard  |495  |
|Mouse     |492  |
|Printer   |488  |
|Monitor   |481  |
|Headphones|481  |
|Speaker   |470  |
+----------+-----+



**MEDALLION ARCHITECTURE**

Bronze state - it is used to store data

Silver state - it is used to clean the data

Gold state - contains bussiness logic,ml, aggregate functions

PySpark is like pandas

In [ ]:
# STEP 8 : Query 1

top_products = df_silver \
    .groupBy('product') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.avg('revenue').alias('avg_order_revenue')
    ) \
    .orderBy('total_revenue', ascending=False) \
    .limit(5)

print('=== Top 5 products By Revenue ===')
top_products.show(truncate=False)

=== Top 5 products By Revenue
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [ ]:
revenue_region = df_silver \
     .groupBy('region') \
     .agg(
         F.sum('revenue').alias('total_revenue'),
         F.count('order_id').alias('num_orders'),
         F.avg('revenue').alias('avg_order_revenue')
     ) \
     .orderBy('total_revenue', ascending=False) \
     .limit(5)

print('=== Revenue by Region ===')
revenue_region.show(truncate=False)

=== Revenue by Region ===
+------+-------------+----------+------------------+
|region|total_revenue|num_orders|avg_order_revenue |
+------+-------------+----------+------------------+
|West  |198275600    |2021      |98107.66947055914 |
|South |147145900    |1483      |99221.7801753203  |
|North |99878400     |995       |100380.30150753769|
|East  |50547700     |501       |100893.6127744511 |
+------+-------------+----------+------------------+

